# Spring of Code - Artificial Intelligence

## Week 09: Deep Learning Advanced

### Day 04: Attention Mechanisms and Transformers

Welcome to the foundation of modern Artificial Intelligence! Today we are learning about **Attention Mechanisms** and **Transformers**, the exact architecture powering models like ChatGPT, BERT, and modern AI.

In this highly detailed notebook, we will go from building the mathematical core of Attention from scratch, all the way to fine-tuning our own Large Language Model (GPT-2)!


### 1. Environment Setup
We will need the Hugging Face `transformers` library, which is the industry standard for working with NLP models.


In [ ]:
!pip install transformers datasets python-docx

import tensorflow as tf
import numpy as np
print(f"TensorFlow Version: {tf.__version__}")

--- 
## Part 1: "Under the Hood" - The Math of Self-Attention
Before we use pre-built models, let's understand the math. The core of a Transformer is **Scaled Dot-Product Attention**.

Formula: $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

Let's pretend we have a tiny sentence with 3 words, and each word is represented by a 4-dimensional vector.


In [ ]:
np.random.seed(42)

# 3 words, 4 dimensions each
sequence_length = 3
d_k = 4  # dimension of keys/queries

# Let's generate random numbers to represent our Query (Q), Key (K), and Value (V) matrices
Q = tf.constant(np.random.rand(sequence_length, d_k), dtype=tf.float32)
K = tf.constant(np.random.rand(sequence_length, d_k), dtype=tf.float32)
V = tf.constant(np.random.rand(sequence_length, d_k), dtype=tf.float32)

print("Query Matrix (Q):\n", Q.numpy())

### Step A: The Dot Product ($Q \times K^T$)
We multiply the Queries by the Keys. This calculates a "score" of how much each word should care about every other word.


In [ ]:
# Matrix multiplication of Q and Transpose of K
raw_scores = tf.matmul(Q, K, transpose_b=True)
print("Raw Attention Scores:\n", raw_scores.numpy())

### Step B: Scaling and Softmax
To prevent the scores from getting too large, we divide by the square root of the dimension size ($\sqrt{d_k}$). Then we apply **Softmax** so the scores for each word add up to 1.0 (100% attention budget).


In [ ]:
# Scale
scaled_scores = raw_scores / tf.math.sqrt(tf.cast(d_k, tf.float32))

# Softmax
attention_weights = tf.nn.softmax(scaled_scores, axis=-1)
print("Attention Weights (Notice each row sums to 1.0):\n", attention_weights.numpy())

### Step C: Multiply by Values ($V$)
Finally, we multiply our attention weights by the Values matrix to get our new, context-aware word vectors.


In [ ]:
context_vectors = tf.matmul(attention_weights, V)
print("Final Context-Aware Output:\n", context_vectors.numpy())

--- 
## Part 2: Multi-Head Attention in Keras
Now that you know the math, you never have to write it from scratch again! TensorFlow provides `tf.keras.layers.MultiHeadAttention`, which does all of the above, but splits the learning across multiple "heads" to learn different types of relationships simultaneously.


In [ ]:
from tensorflow.keras.layers import MultiHeadAttention, Input

# Create a Multi-Head Attention layer with 2 heads
mha_layer = MultiHeadAttention(num_heads=2, key_dim=4)

# In Self-Attention, the sequence acts as its own Query, Key, and Value!
dummy_sequence = tf.random.normal(shape=(1, 3, 4)) # (batch_size, seq_len, dim)

# Pass the sequence in as query, value, and key
attn_output = mha_layer(query=dummy_sequence, value=dummy_sequence, key=dummy_sequence)

print("Multi-Head Attention Output Shape:", attn_output.shape)
print("Notice how the Keras layer handled all the math automatically!")

### Visualizing Attention Heads
A huge part of understanding Transformers is visualizing *where* the model is looking. By asking Keras to `return_attention_scores=True`, we can plot a heatmap showing exactly how strongly each word pays attention to the other words across different heads!

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# 1. Get Attention Scores
attn_output, attn_scores = mha_layer(
    query=dummy_sequence, 
    value=dummy_sequence, 
    key=dummy_sequence, 
    return_attention_scores=True
)

head_matrix = attn_scores[0, 0, :, :].numpy()  # We will look at Head 1

# 2. Set up the figure
fig, ax = plt.subplots(figsize=(8, 5))
words = ["Word 1", "Word 2", "Word 3"]

# Define X and Y coordinates for the left (Query) and right (Key) columns
left_x, right_x = 0, 1
y_coords = [2, 1, 0] 

# 3. Draw the words on the screen
for i, word in enumerate(words):
    # Query words (Attending FROM)
    ax.text(left_x, y_coords[i], word, ha='right', va='center', fontsize=14, fontweight='bold')
    # Key words (Attended TO)
    ax.text(right_x, y_coords[i], word, ha='left', va='center', fontsize=14, fontweight='bold')

ax.text(left_x, 2.8, "Query\n(Attending From)", ha='right', fontsize=12, color='gray')
ax.text(right_x, 2.8, "Key\n(Attended To)", ha='left', fontsize=12, color='gray')

# 4. Draw connection arrows based on attention weights!
for q_idx in range(3):       # For each Query word
    for k_idx in range(3):   # For each Key word
        weight = head_matrix[q_idx, k_idx]
        
        # Only draw arrows if the attention is strong enough
        if weight > 0.01:
            # The arrow points from the Query to the Key
            con = patches.ConnectionPatch(
                xyA=(left_x + 0.05, y_coords[q_idx]), coordsA=ax.transData,
                xyB=(right_x - 0.05, y_coords[k_idx]), coordsB=ax.transData,
                arrowstyle="-|>,head_width=0.4,head_length=0.6", 
                lw=weight * 8,       # Thicker line = stronger attention
                alpha=weight,        # More opaque = stronger attention
                color="#1f77b4",   # Nice blue color
                connectionstyle="arc3,rad=0.1" # Slight curve to the arrows
            )
            ax.add_patch(con)

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 3.5)
ax.axis('off')
plt.title("Attention Flow Visualization (Head 1)", fontsize=16)
plt.show()

--- 
## Part 3: Industry Standard Pre-trained Models (Hugging Face)
In the real world, we rarely build Transformers from scratch. We load massively pre-trained models from **Hugging Face** using the `pipeline` API. Let's see 3 powerful examples!


### 1. Sentiment Analysis


In [ ]:
from transformers import pipeline

# Load a lightweight BERT model for sentiment analysis
sentiment_classifier = pipeline("sentiment-analysis")

result = sentiment_classifier("I absolutely love learning about Artificial Intelligence at NAVTTC!")
print("Sentiment Result:", result)

### 2. Zero-Shot Classification
This is a magical capability. We can give the model a text, and give it custom categories it has *never been explicitly trained on*, and it will classify the text accurately.


In [ ]:
zero_shot_classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

sequence = "The new smartphone has a beautiful OLED display and a massive 5000mAh battery."
candidate_labels = ["politics", "technology", "sports", "cooking"]

result = zero_shot_classifier(sequence, candidate_labels)
print("Zero-Shot Result:")
for label, score in zip(result['labels'], result['scores']):
    print(f"{label}: {score*100:.2f}%")

### 3. Extractive Question Answering
Provide a context paragraph, ask a question, and the Transformer will locate the exact answer.


In [ ]:
qa_pipeline = pipeline("question-answering")

context_text = """
NAVTTC (National Vocational and Technical Training Commission) is an organization in Pakistan 
that provides technical and vocational training. The AI course is designed to equip students 
with modern skills like Deep Learning, Computer Vision, and Natural Language Processing.
"""

question = "What skills does the AI course teach?"

result = qa_pipeline(question=question, context=context_text)
print(f"Question: {question}")
print(f"Answer: {result['answer']}")
print(f"Confidence: {result['score']*100:.2f}%")

--- 
## Part 4: Fine-Tuning a Large Language Model (GPT-2)
Finally, let's do something advanced: Fine-tuning GPT-2 for **Next Word Prediction** on a custom dataset.

To ensure this runs quickly on your computer, we will generate a very tiny "toy" dataset of custom sentences. We will teach GPT-2 facts about NAVTTC.


### Step A: Create the Custom Dataset

In [ ]:
custom_text = """
NAVTTC AI course is the best place to learn deep learning.
Students at NAVTTC learn about transformers and attention mechanisms.
Artificial Intelligence will change the future of technology.
GPT-2 is a powerful language model created by OpenAI.
The instructor at NAVTTC teaches python, tensorflow, and machine learning.
Transformers rely on self-attention to understand the context of words.
NAVTTC students are building the future of AI in Pakistan.
"""

# Save to a text file
with open("navttc_data.txt", "w") as f:
    f.write(custom_text)
    
print("Custom dataset saved!")

### Step B: Tokenization
We need to convert words into numbers using the GPT-2 Tokenizer.


In [ ]:
from transformers import GPT2Tokenizer

# Load the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT2 doesn't have a padding token by default, so we set it to the End Of String token
tokenizer.pad_token = tokenizer.eos_token

# Read and tokenize our text file
with open("navttc_data.txt", "r") as f:
    lines = f.readlines()

# Tokenize the lines
tokenized_data = tokenizer(lines, return_tensors="tf", max_length=32, padding="max_length", truncation=True)

# For Next Word Prediction (Causal Language Modeling), the labels are simply the input_ids shifted!
input_ids = tokenized_data["input_ids"]
attention_mask = tokenized_data["attention_mask"]
labels = tf.identity(input_ids)

print("Data tokenized successfully! Shape:", input_ids.shape)

### Step C: Load the Pre-Trained GPT-2 Model
We use `TFGPT2LMHeadModel` because we are using TensorFlow (`TF` prefix) and doing Language Modeling (`LMHead`).


In [ ]:
from transformers import TFGPT2LMHeadModel

# Load the native TensorFlow GPT-2 model
model = TFGPT2LMHeadModel.from_pretrained("gpt2")

print("Model loaded!")

### Step D: Baseline Inference (Before Fine-Tuning)
Let's see what GPT-2 predicts *before* we train it on our custom data.


In [ ]:
def generate_text(prompt, model, tokenizer):
    input_tensor = tokenizer.encode(prompt, return_tensors="tf")
    output = model.generate(input_tensor, max_length=30, num_return_sequences=1, 
                            pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = "NAVTTC students are"
print("BEFORE FINE-TUNING:")
print(generate_text(prompt, model, tokenizer))

### Step E: Fine-Tune the Model
We compile the model with Adam and fit it! We use `from_logits=True` because the model outputs raw un-normalized scores (logits).


In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=3e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer=optimizer, loss=[loss])

print("Starting Fine-Tuning...")
# We train for 15 epochs so it memorizes our tiny dataset quickly
model.fit([input_ids, attention_mask], labels, epochs=15, batch_size=2)
print("Fine-Tuning Complete!")

### Step F: Evaluate (After Fine-Tuning)
Now let's ask the same prompt. It should generate text heavily influenced by our custom `navttc_data.txt` file!


In [ ]:
print("AFTER FINE-TUNING:")
print(generate_text(prompt, model, tokenizer))

---
## Extension: Fine-Tuning on an MS Word Document (.docx)
For real-world scenarios, your training data might live in an MS Word document rather than a simple text file. Here is how you can use the `python-docx` library to extract text and prepare it for fine-tuning just like we did above.

In [ ]:
import docx
import os

# Let's create a quick sample document to simulate a real scenario
word_file_path = "sample_training_data.docx"
if not os.path.exists(word_file_path):
    doc = docx.Document()
    doc.add_heading('Training Data', 0)
    doc.add_paragraph("This is the first sentence extracted from the MS Word document.")
    doc.add_paragraph("The transformer architecture uses attention to understand sequence data.")
    doc.save(word_file_path)

# 1. Read from the Word Document
doc = docx.Document(word_file_path)
docx_lines = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

print(f"Loaded {len(docx_lines)} paragraphs from the Word file!")

# 2. Tokenize it exactly like before
docx_tokenized = tokenizer(docx_lines, return_tensors="tf", max_length=32, padding="max_length", truncation=True)
print("MS Word Data Tokenized successfully! Shape:", docx_tokenized["input_ids"].shape)

# You could now pass docx_tokenized["input_ids"] into model.fit() just like we did in Step E!